# Stage 1 — Baseline Analysis

This notebook validates the Stage 1 pipeline using GPT-2 on a small sample.
All logic here will be re-run with Mistral-7B for official experiment results.

**What we test:**
1. `metrics.py` — `measure_latency()` and `compute_perplexity()`
2. `full_cache.py` — single inference run
3. `benchmark.py` — full pipeline with WikiText-103

## 1. Setup

In [ ]:
# Clone repo and install dependencies (run once per Colab session)
!git clone https://github.com/yanghao13111/adaptive-kv-cache.git 2>/dev/null || echo "Repo already cloned"
%cd adaptive-kv-cache
!git pull
!pip install transformers accelerate datasets -q

In [ ]:
import torch
import sys
sys.path.insert(0, '/content/adaptive-kv-cache')

from transformers import AutoModelForCausalLM, AutoTokenizer

# Load GPT-2 for validation — swap to mistralai/Mistral-7B-v0.1 for official experiments
MODEL_NAME = "gpt2"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float16).to(DEVICE)
model.eval()
model.generation_config.pad_token_id = tokenizer.eos_token_id
print(f"Model loaded: {MODEL_NAME}")

## 2. Validate `metrics.py`

Test `measure_latency()` and `compute_perplexity()` in isolation.

In [ ]:
from src.eval.metrics import measure_latency, compute_perplexity

# measure_latency — expects latency_ms_per_token, throughput, peak_memory_gb
latency_result = measure_latency(model, tokenizer, "The quick brown fox", max_new_tokens=50, device=DEVICE)
print("measure_latency() result:")
for k, v in latency_result.items():
    print(f"  {k}: {round(v, 4)}")

In [ ]:
# compute_perplexity — use real varied text to avoid artificially low scores
texts = [
    "The history of artificial intelligence began in antiquity with myths and stories of artificial beings. "
    "The modern field of AI research was founded at a workshop held on the campus of Dartmouth College "
    "during the summer of 1956. The attendees became the founders and leaders of AI research for decades. "
    "Many of them predicted that machines as intelligent as humans would exist within a generation. "
    "Optimism was widespread, but progress was slower than expected."
]

ppl = compute_perplexity(model, tokenizer, texts, device=DEVICE, max_length=1024)
print(f"compute_perplexity() result: {ppl:.2f}")
print("(Expected range for GPT-2 on clean English text: 20-60)")

## 3. Validate `full_cache.py`

Test a single inference run and confirm all output keys are present.

In [ ]:
from src.baseline.full_cache import run_full_cache

result = run_full_cache(model, tokenizer, "Once upon a time", max_new_tokens=50, device=DEVICE)

print("run_full_cache() result:")
print(f"  generated_text        : {result['generated_text']}")
print(f"  latency_ms_per_token  : {round(result['latency_ms_per_token'], 2)} ms/token")
print(f"  throughput_tokens_per_sec: {round(result['throughput_tokens_per_sec'], 1)} tok/s")
print(f"  peak_memory_gb        : {round(result['peak_memory_gb'], 3)} GB")

## 4. Run Full Benchmark

Run the end-to-end pipeline with WikiText-103.
`num_samples=5` for quick validation — increase to 50-100 for official results.

In [ ]:
from src.eval.benchmark import run_benchmark, save_results

config = {
    "method": "full_cache",
    "model": MODEL_NAME,
    "dataset": "wikitext-103",
    "num_samples": 5,
    "max_new_tokens": 50,
    "context_len": 1024,
}

results = run_benchmark(config, model, tokenizer)

print("\n=== Benchmark Results ===")
for k, v in results.items():
    print(f"  {k}: {v}")

save_results(results)

## 5. Results Summary

Load the saved CSV and display a summary table.

In [ ]:
import pandas as pd
from pathlib import Path

# Load all result CSVs from experiments/results/
result_files = sorted(Path("experiments/results").glob("*.csv"))

if not result_files:
    print("No results found — run Section 4 first.")
else:
    df = pd.concat([pd.read_csv(f) for f in result_files], ignore_index=True)
    display(df)